# 01 — Exploratory Data Analysis (EDA)
### Climate-Smart Agriculture: Bi-Seasonal Paddy Yield Forecasting
**Module:** IT41033 Nature Inspired Algorithms (NIA)  
**Authors:** Dasun Manjitha, Nimsara, Dinith Sasanga, Wijesinghe, Wijesuriya (Horizon Campus)

---
This notebook explores the historical bi-seasonal paddy production, climate metrics, and macroeconomic indicators in Sri Lanka across ~70 years (1950–2024).

In [ ]:
import sys
import os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import load_primary

# Plot styling
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({'figure.figsize': (12, 6), 'figure.dpi': 150})
SEASON_COLORS = {'Yala': '#E07A3A', 'Maha': '#3A7EBF'}
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Primary Datasets

In [ ]:
datasets = load_primary()
kaggle = datasets['kaggle_rice_climate']
dcs_yield = datasets['dcs_paddy_yield_district']
dcs_extent = datasets['dcs_paddy_extent_district']
dcs_national = datasets['dcs_paddy_extent_national']

print(f"Kaggle Dataset Shape: {kaggle.shape}")
print(f"DCS District Yield Shape: {dcs_yield.shape}")
print(f"DCS District Extent Shape: {dcs_extent.shape}")
print(f"DCS National Extent Shape: {dcs_national.shape}")

## 2. Dataset Inspection & Summary Statistics

In [ ]:
display(kaggle.head(10))
display(kaggle.describe())

## 3. Univariate Distributions by Season (Yala vs Maha)

In [ ]:
continuous_cols = ['rainfall_mm', 'temperature_c', 'production_000_mt',
                   'sown_000_acres', 'harvested_000_acres',
                   'gdp_billion_usd', 'inflation_pct']
col_labels = {
    'rainfall_mm': 'Rainfall (mm)',
    'temperature_c': 'Temperature (°C)',
    'production_000_mt': 'Production (000 Mt)',
    'sown_000_acres': 'Sown (000 Acres)',
    'harvested_000_acres': 'Harvested (000 Acres)',
    'gdp_billion_usd': 'GDP (Billion USD)',
    'inflation_pct': 'Inflation (%)',
}

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    ax = axes[i]
    for season, color in SEASON_COLORS.items():
        data = kaggle[kaggle['season'] == season][col].dropna()
        ax.hist(data, bins=20, alpha=0.6, color=color, label=season, edgecolor='white')
    ax.set_title(col_labels.get(col, col))
    ax.set_xlabel(col_labels.get(col, col))
    ax.set_ylabel('Frequency')
    ax.legend()

# Season record count
ax = axes[7]
season_counts = kaggle['season'].value_counts()
bars = ax.bar(season_counts.index, season_counts.values,
              color=[SEASON_COLORS.get(s, '#888') for s in season_counts.index],
              edgecolor='white', linewidth=1.5)
ax.set_title('Record Count by Season')
ax.set_ylabel('Count')
for bar, val in zip(bars, season_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontweight='bold')

axes[8].set_visible(False)
fig.suptitle('Univariate Analysis — Kaggle Rice & Climate Dataset', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(FIGURES_DIR / '01_univariate_histograms.png')
plt.show()

## 4. Outlier Detection via IQR (Box Plots)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
outlier_report = {}

for i, col in enumerate(continuous_cols):
    ax = axes[i]
    data = kaggle[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((data < lower) | (data > upper)).sum()
    outlier_report[col] = n_outliers
    
    sns.boxplot(data=kaggle, x='season', y=col, ax=ax, palette=SEASON_COLORS, width=0.5)
    ax.set_title(f'{col_labels.get(col, col)}\n({n_outliers} outliers)')
    ax.set_xlabel('')

axes[7].set_visible(False)
fig.suptitle('Outlier Detection — Box Plots with IQR Method', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(FIGURES_DIR / '02_outlier_boxplots.png')
plt.show()

print("Outlier counts (1.5*IQR):", outlier_report)

## 5. Missing Data & Duplicate Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

missing_kaggle = kaggle.isnull().sum()
colors = ['#2ecc71' if v == 0 else '#e74c3c' for v in missing_kaggle.values]
axes[0].barh(missing_kaggle.index, missing_kaggle.values, color=colors, edgecolor='white')
axes[0].set_title('Kaggle Dataset — Missing Values per Column')
axes[0].set_xlabel('Missing Count')

missing_dcs = dcs_yield.isnull().sum()
colors = ['#2ecc71' if v == 0 else '#e74c3c' for v in missing_dcs.values]
axes[1].barh(missing_dcs.index, missing_dcs.values, color=colors, edgecolor='white')
axes[1].set_title('DCS Yield District — Missing Values per Column')
axes[1].set_xlabel('Missing Count')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_missing_data.png')
plt.show()

print("Kaggle exact duplicates:", kaggle.duplicated().sum())
print("Kaggle (year, season) key duplicates:", kaggle.duplicated(subset=['year', 'season']).sum())

## 6. Time Trends Across 7 Decades (1950–2024)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Production
for season, color in SEASON_COLORS.items():
    mask = kaggle['season'] == season
    axes[0, 0].plot(kaggle.loc[mask, 'year'], kaggle.loc[mask, 'production_000_mt'],
                    color=color, label=season, linewidth=1.5, marker='.', markersize=4)
axes[0, 0].set_title('National Paddy Production Over Time (Kaggle)')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Production (000 Mt)')
axes[0, 0].legend()

# Rainfall
for season, color in SEASON_COLORS.items():
    mask = kaggle['season'] == season
    axes[0, 1].plot(kaggle.loc[mask, 'year'], kaggle.loc[mask, 'rainfall_mm'],
                    color=color, label=season, linewidth=1.5, marker='.', markersize=4)
axes[0, 1].set_title('Seasonal Rainfall Over Time')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Rainfall (mm)')
axes[0, 1].legend()

# Temperature
for season, color in SEASON_COLORS.items():
    mask = kaggle['season'] == season
    axes[1, 0].plot(kaggle.loc[mask, 'year'], kaggle.loc[mask, 'temperature_c'],
                    color=color, label=season, linewidth=1.5, marker='.', markersize=4)
axes[1, 0].set_title('Seasonal Temperature Over Time')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Temperature (°C)')
axes[1, 0].legend()

# GDP
axes[1, 1].plot(kaggle['year'], kaggle['gdp_billion_usd'], color='#2c3e50', linewidth=1.5, marker='.', markersize=4)
axes[1, 1].set_title('GDP Over Time (Billion USD)')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('GDP ($B)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_time_trends.png')
plt.show()

## 7. Consistency Check: Kaggle vs DCS National

In [ ]:
kaggle_prod = kaggle[['year', 'season', 'production_000_mt']].rename(
    columns={'production_000_mt': 'kaggle_prod'})
dcs_prod = dcs_national[['year', 'season', 'production_000_mt']].rename(
    columns={'production_000_mt': 'dcs_prod'})

merged = pd.merge(kaggle_prod, dcs_prod, on=['year', 'season'], how='inner')
merged['diff_pct'] = ((merged['kaggle_prod'] - merged['dcs_prod']) / merged['dcs_prod'] * 100).abs()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for season, color in SEASON_COLORS.items():
    mask = merged['season'] == season
    axes[0].scatter(merged.loc[mask, 'dcs_prod'], merged.loc[mask, 'kaggle_prod'],
                    color=color, label=season, alpha=0.7, s=30)
max_val = max(merged['dcs_prod'].max(), merged['kaggle_prod'].max())
axes[0].plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='1:1 Perfect Agreement')
axes[0].set_title('Production: Kaggle vs DCS National')
axes[0].set_xlabel('DCS National Production (000 Mt)')
axes[0].set_ylabel('Kaggle Production (000 Mt)')
axes[0].legend()

for season, color in SEASON_COLORS.items():
    mask = merged['season'] == season
    axes[1].plot(merged.loc[mask, 'year'], merged.loc[mask, 'diff_pct'],
                 color=color, label=season, linewidth=1.5, marker='.', markersize=4)
axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='5% Threshold')
axes[1].set_title('Absolute % Difference Over Time')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('|Difference| (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_consistency_check.png')
plt.show()

print(f"Mean absolute difference: {merged['diff_pct'].mean():.2f}%")
print(f"Records with >5% discrepancy: {(merged['diff_pct'] > 5).sum()}")

## 8. Feature Correlation Matrix

In [ ]:
corr_cols = ['rainfall_mm', 'temperature_c', 'gdp_billion_usd', 'inflation_pct',
             'sown_000_acres', 'harvested_000_acres', 'production_000_mt']
corr_labels = [col_labels.get(c, c) for c in corr_cols]

corr_matrix = kaggle[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=corr_labels, yticklabels=corr_labels,
            square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix (Kaggle Dataset)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_correlation_matrix.png')
plt.show()

## 9. Key Findings & Preprocessing Recommendations

### Summary of Findings:
1. **Dataset Completeness**: The Kaggle Sri Lanka Rice Production & Climate dataset contains 149 records spanning 1950–2024 across both Yala (75) and Maha (74) seasons with **zero missing values**.
2. **Kaggle vs DCS Consistency**: National production metrics match Department of Census & Statistics records with a mean difference of only **0.02%** and **zero instances** of >5% deviation, validating the Kaggle set as authoritative.
3. **Outliers**: Outliers detected in GDP (due to exponential economic growth post-2000) and Inflation (macroeconomic spikes in 2022/2023). These will be treated with IQR-based capping (winsorization) rather than deletion to preserve small sample size ($N=149$).
4. **Strong Predictors**: Sown/Harvested extent ($r = 0.90 - 0.92$) and Rainfall ($r = 0.47$) show strong correlation with seasonal production.